# RGCN



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import numpy as np

from torch_geometric.nn import RGCNConv
from torch_geometric.utils import negative_sampling

from sklearn.metrics import roc_auc_score, average_precision_score


In [2]:
# Carregar os itens salvos do outro notebook
checkpoint = torch.load('dados_prontos.pt', weights_only=False)
train_data = checkpoint['train_data']
val_data = checkpoint['val_data']
test_data = checkpoint['test_data']

# Parâmetros baseados no processamento de dados
in_channels = 1024       # Morgan Fingerprints
num_relations = 1317     # Efeitos colaterais
hidden_channels = 64     # Tamanho da camada interna
out_channels = 32        # Tamanho do embedding final da droga

In [3]:
# Configuração de Hiperparâmetros e Dimensões
data = train_data # Conjunto de treino
num_nodes = data.num_nodes # Numero total de drogas
num_relations = num_relations # Numero de tipos de arestas

### Classe Net (cérebro)

    - Começa definindo as portas de entrada (__init__)
    - Cria a primeira camada (conv1) que resume 1024 em 64
    - Cria a segunda camada (conv2) que resume 64 em 32
    - Cria um espaço na memoria pra salvar uma assionatura de cada efeito (rel_distmult)
    - Encode: recebe os parametros, faz a mistura + ReLU (filtra ruídos) e gera o embedding
    - Decode: Recebe um par de drogas, pega os embeddings das duas drogas e calcula o score

In [4]:
class Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations):
        super().__init__()

        # AMPLIFICADOR
        self.lin_input = torch.nn.Linear(in_channels, hidden_channels)
        self.bn1 = torch.nn.BatchNorm1d(hidden_channels)
        
        # ENCODER
        # Primeira camada recebe os 1024 bits e transforma em 64 características
        self.conv1 = RGCNConv(hidden_channels, hidden_channels, num_relations, num_bases=30)

        # Segunda camada refina os 64 dados para um Embedding final de 32 numeros 
        self.conv2 = RGCNConv(hidden_channels, out_channels,   num_relations, num_bases=30)

        # Camada de Dropout (X de chance de desligar um neurônio)
        self.dropout = torch.nn.Dropout(0.2)
        
        # DECODER
        # Parametro DistMult cria uma assinatura para cada um dos 1317 efeitos
        # É uma matriz onde cada liha representa um efeito especifico
        self.rel_distmult = nn.Parameter(torch.Tensor(num_relations, out_channels))

        # Para os pesos não ficarem zerados
        nn.init.xavier_uniform_(self.rel_distmult)

    def encode(self, x, edge_index, edge_type):
        """Transforma a química e as conexões em um vetor de identidade (Embedding)"""
        
        x = self.lin_input(x)
        x = self.bn1(x).relu() # Força o sinal a aparecer
        x = self.dropout(x)
        
        # Passa pela primeira camada e aplica a função de ativação ReLU
        x = self.conv1(x, edge_index, edge_type).relu()
        x = self.dropout(x)
        # Passa pela segunda camada para gerar o embedding final
        x = self.conv2(x, edge_index, edge_type)
        return F.normalize(x, p=2, dim=-1)

    def decode(self, z, edge_label_index, edge_label_type):
        """Calcula a probabilidade de uma interação existir entre duas drogas"""
        # z: são os embeddings de todas as drogas calculados no encode
        
        # Pegamos os embeddings das drogas que queremos testar (Pares de Origem e Destino)
        u = z[edge_label_index[0]] # Droga A
        v = z[edge_label_index[1]] # Droga B
        
        # Pegamos a assinatura do efeito colateral que estamos testando
        rel = self.rel_distmult[edge_label_type]
        
        # Fórmula DistMult: Multiplica a Droga A pela Assinatura do Efeito e pela Droga B.
        # Se o resultado for alto, a chance de interação é alta.
        return (u * rel * v).sum(dim=-1)
            

In [5]:
# Dispositivo com prioridade em GPU/CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Rodando em: {device}")

Rodando em: cuda


In [6]:
# Normalizando os numeros ADICAO 
train_data.x = torch.nn.functional.normalize(train_data.x, p=2, dim=-1)
val_data.x = torch.nn.functional.normalize(val_data.x, p=2, dim=-1)
test_data.x = torch.nn.functional.normalize(test_data.x, p=2, dim=-1)

# Criando o modelo
model = Net(in_channels=1024, 
            hidden_channels=64, # 64 , 32 | 128 , 64 | 64 , 64 |
            out_channels=64, 
            num_relations=1317).to(device)

# Configurando o Otimizador
# Aqui utilizaremos o Adam
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=0.0001) 
# valores do otimizador: 0.1, 0.0005 | 0.0005 , 0.0001 | 0.0001 , 0.001 | 0.0002 , 0.001 | 0.0001 , 0.0001

In [7]:
def train():
    # Inicia o modelo no modo de treino
    model.train()
    optimizer.zero_grad()

    # Encode (Gera os embeddings)
    z = model.encode(train_data.x, train_data.edge_index, train_data.edge_type)
    pos_edge_index  = train_data.edge_label_index
    edge_label_type = train_data.edge_label_type.to(device)
    
    # Negative Sampling
    num_neg_samples = pos_edge_index.size(1)
    neg_edge_index = torch.stack([
        torch.randint(0, num_nodes, (num_neg_samples,), device=device),
        torch.randint(0, num_nodes, (num_neg_samples,), device=device),
    ], dim=0)

    perm = torch.randperm(edge_label_type.size(0), device=device)[:num_neg_samples]
    neg_edge_type = edge_label_type[perm]

    # Decode (Calcula o score para as arestas reais e falsas)
    pos_out = model.decode(z, pos_edge_index, edge_label_type)
    neg_out = model.decode(z, neg_edge_index, neg_edge_type)

    # Loss (Calcula o erro)
    out = torch.cat([pos_out, neg_out])
    label = torch.cat([
        torch.ones(pos_out.size(0)),
        torch.zeros(neg_out.size(0))
    ]).to(device)
    
    loss = F.binary_cross_entropy_with_logits(out, label) 
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item()


@torch.no_grad()
def test(data_set):
    # Coloca o modelo em modo de avaliação
    model.eval()

    # Gera os embeddings
    z = model.encode(data_set.x, data_set.edge_index, data_set.edge_type)

    # Negative Sampling
    pos_edge_index = data_set.edge_label_index
    edge_label_type = data_set.edge_label_type.to(device)
    num_neg_samples = pos_edge_index.size(1)
    
    # Criando exemplos falsos para o teste 
    neg_edge_index = torch.stack([
        torch.randint(0, num_nodes, (num_neg_samples,), device=device),
        torch.randint(0, num_nodes, (num_neg_samples,), device=device),
    ], dim=0)

    perm = torch.randperm(edge_label_type.size(0), device=device)[:num_neg_samples]
    neg_edge_type = edge_label_type[perm]
    
    # Calculamos o score para os reais e para os falsos
    pos_out = model.decode(z, pos_edge_index, edge_label_type)
    neg_out = model.decode(z, neg_edge_index, neg_edge_type)
    
    # Juntamos tudo 
    out    = torch.cat([pos_out, neg_out]).cpu()
    y_true = torch.cat([
        torch.ones(pos_out.size(0)),
        torch.zeros(neg_out.size(0))
    ]).cpu()
    
    return roc_auc_score(y_true, out)

In [8]:
# Garantindo que os conjuntos estão no mesmo dispositvo
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

## Verificação de Valores

In [9]:
print(f"--- Diagnóstico de train_data.x ---")
print(f"Tem algum valor NaN (nulo)?: {torch.isnan(train_data.x).any()}")
print(f"Tem algum valor Infinito?: {torch.isinf(train_data.x).any()}")
print(f"Valor Máximo: {train_data.x.max().item()}")
print(f"Valor Mínimo: {train_data.x.min().item()}")
print(f"Média dos valores: {train_data.x.mean().item():.4f}")

--- Diagnóstico de train_data.x ---
Tem algum valor NaN (nulo)?: False
Tem algum valor Infinito?: False
Valor Máximo: 1.0
Valor Mínimo: 0.0
Média dos valores: 0.0063


In [10]:
num_rel_real = int(train_data.edge_type.max()) + 1

print(f"Número real de relações detectadas: {num_rel_real}")

Número real de relações detectadas: 1317


In [11]:
# Quantos por cento dos dados são zeros?
zeros = (train_data.x == 0).sum().item()
total = train_data.x.numel()
print(f"Porcentagem de zeros: {(zeros/total)*100:.2f}%")

# Existem colunas que são zero para todos?
colunas_mortas = (train_data.x.sum(dim=0) == 0).sum().item()
print(f"Colunas 'mortas' (todas zero): {colunas_mortas} de {train_data.x.size(1)}")

Porcentagem de zeros: 95.69%
Colunas 'mortas' (todas zero): 0 de 1024


In [12]:
# Verifica se existem linhas duplicadas no train_data.x
unique_rows = torch.unique(train_data.x, dim=0)
print(f"Drogas totais: {train_data.x.size(0)}")
print(f"Drogas com 'DNA' único: {unique_rows.size(0)}")

Drogas totais: 645
Drogas com 'DNA' único: 645


In [13]:
import matplotlib.pyplot as plt

# Conta quantas vezes cada relação aparece
rel_counts = torch.bincount(train_data.edge_type)
print(f"Relação mais comum aparece: {rel_counts.max()} vezes")
print(f"Relação mais rara aparece: {rel_counts.min()} vezes")
print(f"Quantas relações aparecem menos de 10 vezes? {(rel_counts < 10).sum().item()}")

Relação mais comum aparece: 22310 vezes
Relação mais rara aparece: 0 vezes
Quantas relações aparecem menos de 10 vezes? 26


In [14]:
print(f"pos_edge_index shape:  {train_data.edge_label_index.shape}")
print(f"edge_label_type shape: {train_data.edge_label_type.shape}")

pos_edge_index shape:  torch.Size([2, 1756101])
edge_label_type shape: torch.Size([1756101])


In [21]:

print(f"{'Época':<7} | {'Loss (Erro)':<12} | {'Val AUC':<10} | {'Progresso'}")
print("-" * 50)

melhor_auc = 0.0
inicio = 201
fim = 301
#Loop principal
for epoch in range(inicio, fim):
    
    loss = train()
    auc = test(val_data)
    
    print(f"{epoch:03d}     | {loss:.4f}      | {auc:.4f}     | {auc*100:>5.1f}%")

    # Salva o melhor modelo
    if auc > melhor_auc:
        melhor_auc = auc
        torch.save(model.state_dict(), 'melhor_modelo.pt')

print("-" * 50)
print("Treinamento finalizado!")

Época   | Loss (Erro)  | Val AUC    | Progresso
--------------------------------------------------
201     | 0.6644      | 0.8638     |  86.4%
202     | 0.6650      | 0.8669     |  86.7%
203     | 0.6645      | 0.8753     |  87.5%
204     | 0.6639      | 0.8751     |  87.5%
205     | 0.6644      | 0.8754     |  87.5%
206     | 0.6636      | 0.8707     |  87.1%
207     | 0.6638      | 0.8710     |  87.1%
208     | 0.6637      | 0.8752     |  87.5%
209     | 0.6633      | 0.8764     |  87.6%
210     | 0.6633      | 0.8753     |  87.5%
211     | 0.6631      | 0.8748     |  87.5%
212     | 0.6629      | 0.8720     |  87.2%
213     | 0.6629      | 0.8736     |  87.4%
214     | 0.6627      | 0.8754     |  87.5%
215     | 0.6625      | 0.8760     |  87.6%
216     | 0.6626      | 0.8754     |  87.5%
217     | 0.6623      | 0.8743     |  87.4%
218     | 0.6622      | 0.8738     |  87.4%
219     | 0.6621      | 0.8756     |  87.6%
220     | 0.6619      | 0.8750     |  87.5%
221     | 0.6619     

In [22]:
# Testando com os de teste
model.load_state_dict(torch.load('melhor_modelo.pt'))
auc_final = test(test_data)
print(f"AUC final no test: {auc_final:.4f}")

AUC final no test: 0.8787


# Referências
    - https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.RGCNConv.html#torch_geometric.nn.conv.RGCNConv
    - https://huggingface.co/riship-nv/RGCN
    - https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html
    - https://www.youtube.com/watch?v=MWZakqZDgfQ
    - https://docs.pytorch.org/ignite/generated/ignite.metrics.ROC_AUC.html